Wiki_tools is the general package name. web_queries are functions I wrote to connect to Wikidata's servers and run specialized queries on their DB.<br>
Specify a query to get keys from - write to csv.
Then process the keys in batches based on what information we need from them.

In [2]:
import wiki_tools.web_queries
import csv
import json

Query is a SPARQL (Wikipedia's SQL implementation) query on their database.<br>
SELECT statements query for certain variables, instead of columns.<br>
The first step is to get the IDs of all of the entities we want. The Query Service can't return everything else efficiently. <br>

In [ ]:
query_body = """
    ?item wdt:P31 wd:Q18127 ;
            wdt:P136 wd:Q8341 .
"""
query = f"""
    SELECT ?item
    WHERE
    {{
        {query_body}
    }}
"""
print("API request:", query)

Specify a file path here.

In [ ]:
file_name = "label_ids.csv"

In [ ]:
ids = wiki_tools.web_queries.get_ids(query)
if ids:
    with open(file_name, 'w', newline='') as writing_to:
        writer = csv.writer(writing_to)
        writer.writerow(['id'])
        for row in ids:
            writer.writerow([row])

Now, request the information about each entity with the MediaWiki API, which is far more efficient. <br>
get_batch_data will take in the mode, a file of keys and a csv outfile for further processing. <br>
Note - this will work for a infile that contains either id's OR links themselves as well.

In [ ]:
mode = 'name-only'
infile = 'instrument_ids.csv'
outfile = 'instruments_db.csv'

In [ ]:
wiki_tools.web_queries.get_batch_data(mode, infile, outfile)

Some stats (for a 271 batch file):
- $271 * 50 = 13350$ lines to be read/queried
- Took 25m 17s or $1571s$
- $271 * 5 = 1335s$ of requery waiting
- $1571 - 1335 = 162s$ total working time
- $162sec / 271 queries = .59$ sec/query

Run queries on individual items for debugging.

In [3]:
test_enity = "Q2491445"
test_result = wiki_tools.web_queries.get_page_from_id(test_enity) 
json.dumps(test_result)

'[{"Q2491445": {"pageid": 2405359, "ns": 0, "title": "Q2491445", "lastrevid": 2544950993, "modified": "2026-09-13T08:51:26Z", "type": "item", "id": "Q2491445", "labels": {"en": {"language": "en", "value": "Francien van Tuinen"}}, "descriptions": {"en": {"language": "en", "value": "Dutch singer"}}, "aliases": {}, "claims": {"P434": [{"mainsnak": {"snaktype": "value", "property": "P434", "hash": "039876ca130db2595ee42becfa901da862af384f", "datavalue": {"value": "04e73054-cf0e-4299-b97e-dee1d98b2516", "type": "string"}, "datatype": "external-id"}, "type": "statement", "id": "q2491445$4D423A1B-3BA3-4AA8-BE9E-D67C5FB2EF99", "rank": "normal", "references": [{"hash": "706208b3024200fd0a39ad499808dd0d98d74065", "snaks": {"P248": [{"snaktype": "value", "property": "P248", "hash": "623cc8f0e2f65afe4d66b91962d354a2f3aa9a27", "datavalue": {"value": {"entity-type": "item", "numeric-id": 14005, "id": "Q14005"}, "type": "wikibase-entityid"}, "datatype": "wikibase-item"}]}, "snaks-order": ["P248"]}]}]

Compiling a quick list of the unique instruments found in the musicians_db.

In [ ]:
instrument_list = []
with open("played.csv", 'r') as f:
    reader = csv.DictReader(f)
    for row in reader:
        instrument = row['inst']
        if instrument in instrument_list:
            continue
        else:
            instrument_list.append(instrument)

with open("instrument_ids.csv", 'w') as f:
    writer = csv.writer(f)
    for instrument in instrument_list:
        writer.writerow([instrument])